# 02 — Position Structures · Task 14.3

Sample realistic Uniswap v3 LP positions `(Pa, Pb, P0, L, V0)` to feed the IL formulas in notebook 03.

Mixture model (placeholder weights — calibrated in Phase 14.7 against on-chain Uniswap v3 data):

| Bucket | Half-width | Weight |
|---|---|---|
| tight | ±5% | 30% |
| moderate | ±15% | 40% |
| wide | ±30% | 25% |
| v2-like | ±80% | 5% |

Each draw also gets a log-normal V0 (notional at creation) and a centred-Beta offset (where P0 sits in [Pa, Pb]).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from inflexion_quant.positions import PositionMix, sample_positions

rng = np.random.default_rng(seed=20260526)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': 0.3})

## A. Sample 10k positions and inspect

In [ ]:
mix = PositionMix.crypto_majors()
df = sample_positions(10_000, P0=3000.0, mix=mix, rng=rng)
print(f'sampled {len(df):,} positions')
print(df.head())

print('\nbucket frequencies (vs target):')
tbl = pd.DataFrame({
    'target': np.asarray(mix.weights) / sum(mix.weights),
    'sampled': df['bucket'].value_counts(normalize=True).sort_index().values,
    'half_width_target': mix.half_widths,
})
tbl.index.name = 'bucket'
print(tbl.round(3))

## B. Geometry + V0 distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['half_width'], bins=80, color='C0', alpha=0.8)
axes[0].set(xlabel='half-width (fraction of P0)', ylabel='count',
            title='range half-width (mixture: 4 buckets, jittered)', xscale='log')

axes[1].hist(df['offset_fraction'], bins=60, color='C2', alpha=0.8)
axes[1].set(xlabel='offset_fraction', ylabel='count',
            title='P0 offset in [Pa,Pb] (0 = centred, ±1 = boundary)')

axes[2].hist(np.log10(df['V0']), bins=60, color='C3', alpha=0.8)
axes[2].set(xlabel='log10(V0)  [USD]', ylabel='count',
            title=f'notional V0 (median ${df.V0.median():,.0f}, mean ${df.V0.mean():,.0f})')
plt.tight_layout(); plt.show()

## C. Sanity — Pa, Pb in price space

In [ ]:
# Show 200 random positions as horizontal range bars vs P0
P0 = 3000.0
sample = df.sample(200, random_state=42).sort_values('half_width').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 6))
for i, row in sample.iterrows():
    ax.plot([row['Pa'], row['Pb']], [i, i], color='C0', alpha=0.5, lw=1.2)
    ax.plot([row['P0']], [i], 'o', color='k', markersize=2)
ax.axvline(P0, color='r', ls='--', lw=0.8, alpha=0.6, label='entry price P0')
ax.set(xlabel='price (USD)', ylabel='position (sorted by half-width)',
       title='200 sampled positions — sorted tight→wide; dot = P0', xscale='log')
ax.legend(); plt.tight_layout(); plt.show()

## D. Next

Notebook **03_path_to_il.ipynb** (Task 14.4) takes these positions, runs them against the simulated price paths from notebook 01, and computes the IL distribution that ultimately feeds the portfolio waterfall (Task 14.5) and the fund-solvency analysis (Task 14.6).